# 04 — QLoRA Configuration Exploration

**Goal:** Understand what LoRA rank and alpha mean by checking trainable parameter counts before committing to a full training run.

**Day 17** of the implementation plan.

> Key insight: LoRA trains ~2 million adapter parameters instead of all 3.8 billion model weights.
> That's why it fits on a 6GB GPU.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

model_name = 'microsoft/Phi-3-mini-4k-instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print('Loading base model...')
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)
print('Done!')

In [ ]:
# Apply our LoRA config
lora_config = LoraConfig(
    r=16,                    # Rank: controls how many parameters the adapter has
    lora_alpha=32,           # Scaling factor (rule of thumb: 2 * r)
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],  # Attention layers
    lora_dropout=0.05,       # Light regularization to prevent overfitting
    bias='none',
    task_type=TaskType.CAUSAL_LM
)

peft_model = get_peft_model(model, lora_config)

# This shows exactly how many parameters we're actually training
peft_model.print_trainable_parameters()
# Expected: trainable params: ~2M || all params: ~3.8B || trainable%: ~0.05%

In [ ]:
# --- Experiment: How does rank affect parameter count? ---
# (We won't actually train these — just checking the math)

print('Effect of different LoRA ranks on trainable parameter count:')
print()

for r in [4, 8, 16, 32, 64]:
    config = LoraConfig(
        r=r,
        lora_alpha=r*2,
        target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
        bias='none',
        task_type=TaskType.CAUSAL_LM
    )
    peft_m = get_peft_model(model, config)
    trainable = sum(p.numel() for p in peft_m.parameters() if p.requires_grad)
    total = sum(p.numel() for p in peft_m.parameters())
    print(f'  r={r:3d} | alpha={r*2:3d} | trainable: {trainable/1e6:.2f}M / {total/1e9:.2f}B ({trainable/total*100:.3f}%)')
    del peft_m  # Free memory between iterations

In [ ]:
# Write in LEARNINGS.md:
# - What does the trainable% tell you?
# - What would happen if you increased the rank to 64?
# - Why do we target q_proj, v_proj, k_proj, o_proj specifically?
print('Done. Write your observations in LEARNINGS.md.')